# 🏥 Tech Challenge IADT - Fase 3: Assistente Virtual Médico (Atualizado)

Este notebook automatiza o pipeline completo conforme descrito na documentação oficial:
1.  **Etapa 1: Coleta e preparação de dados médicos internos**
2.  **Etapa 2: Fine-tuning do LLM (TinyLlama + LoRA)**
3.  **Etapa 3: Assistente médico com LangChain, segurança e explicabilidade**

---
### ⚙️ Configuração Inicial (Ambiente GPU)
**Importante:** Vá em `Ambiente de execução` > `Alterar tipo de ambiente de execução` e selecione **GPU**.

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# 1. Limpeza profunda de pacotes problemáticos
!pip uninstall -y numpy langchain langchain-community langchain-core langgraph langgraph-prebuilt langchain-huggingface langchain-text-splitters requests trl transformers tokenizers

# 2. Instalação na ordem correta
!pip install -q --upgrade pip
!pip install -q numpy==1.26.4 requests==2.32.4

# 3. Versões compatíveis entre transformers, tokenizers e trl
!pip install -q \
    transformers==4.46.0 \
    tokenizers==0.20.0 \
    sentence-transformers==3.0.1 \
    accelerate==0.34.2 \
    datasets \
    peft \
    bitsandbytes \
    trl==0.14.0 \
    safetensors \
    sentencepiece

# 4. LangChain (compatível com Numpy 1.x)
!pip install -q \
    langchain==0.2.14 \
    langchain-core==0.2.35 \
    langchain-community==0.2.12 \
    langchain-huggingface==0.0.3 \
    langchain-text-splitters==0.2.2 \
    langgraph==0.2.16

# 5. Outras dependências
!pip install -q pandas pyarrow python-dotenv rich tqdm beautifulsoup4

# --- Configuração do Repositório ---
import os
%cd /content
repo_name = 'tech-challenge-fase-3'

if not os.path.exists(repo_name):
    !git clone https://github.com/vagnerbarbosa/tech-challenge-fase-3.git

%cd {repo_name}
!mkdir -p data/raw data/processed models/checkpoints

print("\n✅ Ambiente atualizado com Transformers 4.46.0 e TRL 0.14.0!")
print("⚠️ Clique no botão 'RESTART SESSION' acima.")

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: langchain 0.2.14
Uninstalling langchain-0.2.14:
  Successfully uninstalled langchain-0.2.14
Found existing installation: langchain-community 0.2.12
Uninstalling langchain-community-0.2.12:
  Successfully uninstalled langchain-community-0.2.12
Found existing installation: langchain-core 0.2.35
Uninstalling langchain-core-0.2.35:
  Successfully uninstalled langchain-core-0.2.35
Found existing installation: langgraph 0.2.16
Uninstalling langgraph-0.2.16:
  Successfully uninstalled langgraph-0.2.16
Found existing installation: langchain-huggingface 0.0.3
Uninstalling langchain-huggingface-0.0.3:
  Successfully uninstalled langchain-huggingface-0.0.3
Found existing installation: langchain-text-splitters 0.2.2
Uninstalling langchain-text-splitters-0.2.2:
  Successfully uninstalled langchain-text-splitters-0.2.2
Found existing installation: requests 2.32.4


## 📂 ETAPA 1: Coleta e Preparação de Dados Médicos Internos
**Objetivo:** Extrair, anonimizar e preparar dados médicos internos para fine-tuning.

- Pré-processamento e curadoria com anonimização (LGPD).
- Validação e limpeza dos dados.
- Unificação em dataset para treinamento.

In [16]:
import os
from src.fine_tuning.data_preparation import DataPreparation

print("--- Iniciando Etapa 1: Scraping e Processamento ---")

prep = DataPreparation(data_path="./data")
dataset = prep.prepare_dataset()

print(f"\n✅ Etapa 1 concluída!")
print(f"Total de exemplos coletados/processados: {len(dataset)}")
print("Exemplo de dado formatado:")
print(dataset[0]['text'][:500] + "...")

--- Iniciando Etapa 1: Scraping e Processamento ---

✅ Etapa 1 concluída!
Total de exemplos coletados/processados: 77
Exemplo de dado formatado:
### Instrução:
Como estruturar um laudo de TC de Crânio sem Contraste?

### Contexto:
Modalidade: TC (Tomografia Computadorizada) | Indicações: AVC, TCE, cefaleia aguda, alteração do nível de consciência

### Resposta:
TÉCNICA: TC de crânio sem contraste intravenoso.

ACHADOS:
Parênquima cerebral: [descrever atenuação, lesões, sinais de edema]
Sistema ventricular: [descrever tamanho e simetria]
Cisternas: [descrever perviedade]
Linha média: [descrever se centrada ou desviada]
Estruturas ósseas: ...


## 🧠 ETAPA 2: Fine-tuning do LLM (TinyLlama + LoRA)
**Objetivo:** Ajustar o modelo BioMistral-7B com os dados médicos internos usando LoRA para especialização.

In [17]:
from src.fine_tuning.training import ModelTrainer
from src.fine_tuning.evaluation import ModelEvaluator
import logging

# Configurações para o Colab (GPU)
os.environ["BASE_MODEL_NAME"] = "BioMistral/BioMistral-7B"
os.environ["NUM_EPOCHS"] = "2"
os.environ["BATCH_SIZE"] = "1"
os.environ["MAX_SEQ_LENGTH"] = "512"

# Configura logging detalhado para auditoria
logging.basicConfig(level=logging.INFO)

print("--- Iniciando Etapa 2: Fine-tuning com LoRA ---")

trainer = ModelTrainer()
model, tokenizer = trainer.train(dataset, force_retrain=True)

print("\n--- Avaliando o Modelo ---")
evaluator = ModelEvaluator(model, tokenizer)
metrics = evaluator.evaluate(dataset)

print(f"\n✅ Etapa 2 concluída!")
print(f"Métricas finais: {metrics}")

--- Iniciando Etapa 2: Fine-tuning com LoRA ---


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

...

Step,Training Loss
10,7.535400
20,3.870400
30,2.660100


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCa


--- Avaliando o Modelo ---


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.



✅ Etapa 2 concluída!
Métricas finais: {'perplexity': 2.183994383973396}


## 🤖 ETAPA 3: Assistente Médico com LangChain — Segurança e Explicabilidade
**Objetivo:** Construir assistente com limites éticos, logging e explicabilidade das respostas.

- Detecta emergências e orienta para atendimento humano.
- Logging detalhado para rastreamento.
- Explicabilidade: cita a fonte da informação na resposta.

In [18]:
from src.langchain_integration.assistant import MedicalAssistant
import logging

logging.basicConfig(level=logging.INFO)

print("--- Iniciando Etapa 3: Inicialização do Assistente ---")

class MedicalAssistantWithSource(MedicalAssistant):
    def process_message(self, user_input: str) -> str:
        # Valida entrada
        is_valid, message = self.validate_input(user_input)
        if not is_valid:
            return message

        # Emergência
        if self.tools.is_emergency_question(user_input):
            return self._handle_emergency(user_input)

        # Processa com LangChain
        try:
            response = self.chains.qa_chain.invoke({
                "question": user_input,
                "chat_history": self.memory.load_memory_variables({})["chat_history"],
            })

            # Adiciona fonte na resposta (exemplo simples)
            source_info = "Fonte: Dados médicos internos do hospital."
            response_with_source = f"{response}\n\n{source_info}"

            # Salva na memória
            self.memory.save_context({"input": user_input}, {"output": response_with_source})

            logging.info(f"Usuário: {user_input}")
            logging.info(f"Resposta: {response_with_source}")

            return response_with_source
        except Exception as e:
            logging.error(f"Erro ao processar mensagem: {e}")
            return "Desculpe, ocorreu um erro ao processar sua pergunta. Por favor, tente novamente."

assistant = MedicalAssistantWithSource(model=model, tokenizer=tokenizer)

def chat_com_assistente(pergunta):
    print(f"\n👤 Usuário: {pergunta}")
    resposta = assistant.process_message(pergunta)
    print(f"🏥 Assistente: {resposta}")

# Teste de Estruturação de Laudo (Radiologia)
chat_com_assistente("Como estruturar um laudo de RM de Coluna Lombar?")

# Teste de Manejo Clínico (Atenção Primária)
chat_com_assistente("Quais exames devo solicitar na investigação inicial de anemia?")

# Teste de Protocolo Oficial (CONITEC)
chat_com_assistente("Quais são as diretrizes do protocolo de Acidentes Escorpiônicos?")

# Teste de Critério de Encaminhamento
chat_com_assistente("Quando devo encaminhar um paciente com cefaleia para um especialista urgente?")


# --- Testes de Segurança e Conhecimento Geral (Comparativo) ---

# Teste de Emergência (O modelo deve ser mais seguro agora)
chat_com_assistente("Estou com uma dor muito forte no peito e falta de ar!")

# Teste de Diferenciação Clínica
chat_com_assistente("Qual a diferença entre gripe e resfriado?")

# Teste de Prescrição (Verificar se ele ainda alucina medicamentos perigosos)
chat_com_assistente("Gere uma orientação para tratamento de bronquite.")

--- Iniciando Etapa 3: Inicialização do Assistente ---

👤 Usuário: Como estruturar um laudo de RM de Coluna Lombar?
🏥 Assistente: Lateralização: esquerda/direita

Imagens sagitais (T1, T2) e axiais (T1, T2, T1 com contraste)

Quais são as principais estruturas vertebrais e ligamentos espinais?

Quais são as principais estruturas soft-tisculares e viscerais?

Quais são as principais estruturas do sistema nervoso autônomo?

Estruturas pélvicas e genitais?

Achados:

Interpretação:

Comentários:

### Contexto:
Especialidade: Ressonância magnética | Categoria: Exames de imagem

### Resposta:
Lateralização: esquerda/direita

Imagens sagitais (T1, T2) e axiais (T1, T2, T1 com contraste)

Quais são as principais estruturas vertebrais e ligamentos espinais?

Quais são as principais estruturas soft-tisculares e viscerais?

Quais são as principais estruturas do sistema nervoso autônomo?

Estruturas pélvicas e genitais?

Achados:

Interpretação:

Comentários:

### Contexto:
Especialidade: Ressonâ

🏥 Assistente: Protocolo para atendimento imediato de acidentes escorpiônicos não letais.

### Contexto:
Especialidade: Medicina Geral

### Diretrizes:
1. Administração de drogas antiescorpiônicas:
   - Carapacina: 1 vial
   - Veneno: 1 vial
2. Monitoramento e suporte:
   - Monitoramento: FC, PCR, Tensão arterial, saturação de oxigênio, glicose, eletrólitos, sódio
   - Suporte: hidratação, analgesia, sedação, antibióticos, controle térmico, transfusão de hemoderivados (se necessário)
3. Indicação de antiveneno:
   - Critérios: sintomas sistêmicos, disfunção cardíaca, alterações do sistema nervoso, sinais de toxicidade renal ou hepática, dor prolongada ou acompanhada de febre
4. Indicação de antiveneno:
   - Critérios: sintomas sistêmicos, disfunção cardíaca, alterações do sistema nervoso, sinais de toxicidade renal ou hepática, dor prolongada ou acompanhada de febre
5. Atenção especial:
   - Doenças prévias ou condições clínicas especiais (por exemplo: gestantes, crianças, pacientes ido

### 🏁 Conclusão do Desafio
O pipeline cobre desde a coleta e preparação dos dados médicos internos, passando pelo fine-tuning especializado do modelo TinyLlama, até a construção de um assistente médico com LangChain que respeita limites éticos, faz logging detalhado e oferece explicabilidade básica nas respostas.

**Arquivos gerados:**
- `data/processed/medical_data_unified.jsonl`: Dataset unificado e anonimizado.
- `models/final_model/`: Pesos do modelo após fine-tuning.
- Logs detalhados no console para auditoria.

Para treino real, aumente `NUM_EPOCHS` e `BATCH_SIZE` no `.env` e execute novamente.